# Train YOLOv11s for iPhone Damage + Generation Detection

**Dataset:** 12 classes (9 generation + scratch + physical_damage + screen_defect)  
**Model:** YOLOv11s (small variant, ~9.5M params)  
**Hardware:** Google Colab T4 GPU  
**Expected:** ~2-3 hours for 100 epochs, mAP@50 ≥ 0.70 overall

## Steps
1. Upload `yolo_dataset_merged.zip` lên Google Drive (folder `MyDrive/khoa_luan/`)
2. Mount Drive
3. Install ultralytics + extract dataset
4. Train
5. Save `best.pt` về Drive

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Install Ultralytics

In [ ]:
!pip install -q ultralytics==8.3.0
import ultralytics
ultralytics.checks()

## 4. Extract dataset từ Drive

**Trước khi chạy cell này:** zip folder `yolo_dataset_merged/` → upload lên Drive `MyDrive/khoa_luan/yolo_dataset_merged.zip`

In [ ]:
import os, shutil, zipfile

ZIP_PATH = '/content/drive/MyDrive/khoa_luan/yolo_dataset_merged.zip'
EXTRACT_TO = '/content/dataset'

if not os.path.exists(EXTRACT_TO):
    os.makedirs(EXTRACT_TO)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_TO)

# Find data.yaml (handle nested folder)
for root, dirs, files in os.walk(EXTRACT_TO):
    if 'data.yaml' in files:
        DATA_YAML = os.path.join(root, 'data.yaml')
        break

print(f'data.yaml: {DATA_YAML}')
!cat {DATA_YAML}

## 5. Fix data.yaml paths cho Colab

Roboflow export dùng path `../train/images` relative — sửa thành absolute.

In [ ]:
import yaml

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

DATASET_ROOT = os.path.dirname(DATA_YAML)
cfg['train'] = os.path.join(DATASET_ROOT, 'train', 'images')
cfg['val']   = os.path.join(DATASET_ROOT, 'valid', 'images')
cfg['test']  = os.path.join(DATASET_ROOT, 'test', 'images')

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"Classes ({cfg['nc']}): {cfg['names']}")
print(f"Train: {cfg['train']}")
print(f"Val:   {cfg['val']}")
print(f"Test:  {cfg['test']}")

## 6. Train YOLOv11s

**Hyperparams chuẩn cho dataset 11k images, 12 classes:**
- `epochs=100` — đủ để converge, dùng `patience=20` early stop
- `imgsz=640` — match Roboflow resize
- `batch=32` — T4 16GB vừa đủ cho YOLOv11s
- `optimizer='AdamW'` — stable hơn SGD cho object detection
- `cos_lr=True` — cosine LR decay
- `cache=True` — cache image trong RAM (load nhanh)
- `amp=True` — mixed precision FP16 (nhanh + giảm VRAM)

Thời gian dự kiến: **2-3 giờ** trên T4.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')  # tự download pretrained COCO weights

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=32,
    optimizer='AdamW',
    cos_lr=True,
    patience=20,
    cache=True,
    amp=True,
    device=0,
    project='/content/runs',
    name='yolov11s_iphone',
    exist_ok=True,
)

print('\nTraining done!')
print(f'Best weights: {results.save_dir}/weights/best.pt')

## 7. Evaluate trên test set

In [ ]:
best_pt = '/content/runs/yolov11s_iphone/weights/best.pt'
model = YOLO(best_pt)

metrics = model.val(data=DATA_YAML, split='test', imgsz=640)
print(f'\n=== TEST SET METRICS ===')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'\nPer-class mAP@50:')
for i, name in enumerate(cfg['names']):
    print(f'  {name:<20} {metrics.box.maps[i]:.4f}')

## 8. Save best.pt + results về Drive

In [ ]:
import shutil

DRIVE_OUT = '/content/drive/MyDrive/khoa_luan/yolov11s_iphone'
os.makedirs(DRIVE_OUT, exist_ok=True)

# Copy best.pt + last.pt
shutil.copy('/content/runs/yolov11s_iphone/weights/best.pt', f'{DRIVE_OUT}/best.pt')
shutil.copy('/content/runs/yolov11s_iphone/weights/last.pt', f'{DRIVE_OUT}/last.pt')

# Copy training graphs + results
for fname in ['results.png', 'results.csv', 'confusion_matrix.png',
              'confusion_matrix_normalized.png', 'F1_curve.png',
              'PR_curve.png', 'P_curve.png', 'R_curve.png']:
    src = f'/content/runs/yolov11s_iphone/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_OUT}/{fname}')

print(f'Saved to: {DRIVE_OUT}')
!ls -la {DRIVE_OUT}

## 9. Test inference trên 1 ảnh

Sanity check trước khi download về local.

In [ ]:
import random
from PIL import Image

test_dir = cfg['test']
sample = random.choice(os.listdir(test_dir))
img_path = os.path.join(test_dir, sample)

results = model.predict(img_path, imgsz=640, conf=0.25, save=True, project='/content', name='preds')
for r in results:
    print(f'\nImage: {sample}')
    print(f'Detections: {len(r.boxes)}')
    for box in r.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        print(f'  {cfg["names"][cls]:<20} conf={conf:.3f}')

# Display the predicted image
from IPython.display import Image as IPImage
IPImage(f'/content/preds/{sample}')

## 10. Done

**Download `best.pt` từ Drive → đặt vào local:**
```
ai-service-vision/app/models/best.pt
```

**Mục tiêu pass:**
- mAP@50 ≥ 0.70 overall
- mAP@50 ≥ 0.75 cho generation classes
- mAP@50 ≥ 0.55 cho damage classes (physical_damage, scratch, screen_defect)

**Nếu fail target:**
- Check confusion matrix → class nào confused → label thêm 100-200 ảnh
- Tăng epochs lên 150 với patience=30
- Thử YOLOv11m nếu T4 còn vRAM